In [14]:
# Imports and Setup
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:

# Data Pipeline (Direct CSV Download)

url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(url)

# Map 'positive' to 1 and 'negative' to 0
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split into train and test
train_df, test_df = train_test_split(df, train_size=10000, test_size=1000, random_state=42)

# Keep only the text column for training 
train_df = train_df[['review']].rename(columns={'review': 'text'})
test_df = test_df[['review', 'label']].rename(columns={'review': 'text'})

print(f"Loaded {len(train_df)} training samples and {len(test_df)} test samples.")

Loaded 10000 training samples and 1000 test samples.


In [16]:
#Heuristic Rules / Weak Labelling

STRONG_POS = ["masterpiece", "excellent", "amazing", "perfect", "loved it", "brilliant", "best"]
WEAK_POS = ["good", "enjoyable", "decent", "fine", "liked", "entertaining"]
STRONG_NEG = ["terrible", "awful", "worst", "hate", "boring", "waste of time", "horrible"]
WEAK_NEG = ["bad", "poor", "disappointing", "slow", "dull"]

def apply_weak_labels(text):
    text_lower = str(text).lower()

    if any(word in text_lower for word in STRONG_POS): return 1, 1.0
    if any(word in text_lower for word in STRONG_NEG): return 0, 1.0

    if any(word in text_lower for word in WEAK_POS): return 1, 0.5
    if any(word in text_lower for word in WEAK_NEG): return 0, 0.5

    return -1, 0.0

In [17]:
weak_results = train_df['text'].apply(apply_weak_labels)
train_df['weak_label'] = weak_results.apply(lambda x: x[0])
train_df['confidence'] = weak_results.apply(lambda x: x[1])


# Filter out samples that didn't match any rules (label == -1)
train_df = train_df[train_df['weak_label'] != -1].reset_index(drop=True)
print(f"Training samples after weak labeling: {len(train_df)}")

Training samples after weak labeling: 8441


In [ ]:
# TF-IDF Vectorization 

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit on train text and transform
X_train = vectorizer.fit_transform(train_df['text']).toarray()
y_train = train_df['weak_label'].values
w_train = train_df['confidence'].values

# Transform test text
X_test = vectorizer.transform(test_df['text']).toarray()
y_test = test_df['label'].values

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
w_train_tensor = torch.tensor(w_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor, w_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [19]:
#  Custom MLP & Confidence-Weighted Loss

class SimpleMLP(nn.Module):
    def __init__(self, input_dim):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 2) # 2 classes (Pos/Neg)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

class ConfidenceWeightedLoss(nn.Module):
    def __init__(self):
        super(ConfidenceWeightedLoss, self).__init__()

    def forward(self, logits, targets, weights):
        # Calculate standard cross entropy loss without reduction
        loss = torch.nn.functional.cross_entropy(logits, targets, reduction='none')
        # Multiply loss by sample confidence weights and take the mean
        weighted_loss = loss * weights
        return weighted_loss.mean()

criterion = ConfidenceWeightedLoss()



In [20]:

# Training & Evaluation Functions

def train_model(model, loader, use_weights=True):
    optimizer = AdamW(model.parameters(), lr=0.001)
    model.train()

    for epoch in range(10): # 10 epochs since MLP is small
        total_loss = 0
        for inputs, labels, weights in tqdm(loader, desc=f"Epoch {epoch+1}"):
            inputs, labels, weights = inputs.to(device), labels.to(device), weights.to(device)

            optimizer.zero_grad()
            logits = model(inputs)

            if use_weights:
                loss = criterion(logits, labels, weights)
            else:
                loss = torch.nn.functional.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [21]:

# Execution: Baseline vs Robust Model

print("\n--- Training Baseline Model (Ignoring Confidence) ---")
baseline_model = SimpleMLP(input_dim=5000).to(device)
train_model(baseline_model, train_loader, use_weights=False)
baseline_acc = evaluate_model(baseline_model, test_loader)
print(f"Baseline Test Accuracy: {baseline_acc*100:.2f}%\n")

print("\n--- Training Robust Model (Confidence-Weighted Loss) ---")
robust_model = SimpleMLP(input_dim=5000).to(device)
train_model(robust_model, train_loader, use_weights=True)
robust_acc = evaluate_model(robust_model, test_loader)
print(f"Robust Test Accuracy: {robust_acc*100:.2f}%\n")

print("="*40)
print("FINAL SUMMARY:")
print(f"Baseline (Standard Loss): {baseline_acc*100:.2f}%")
print(f"Robust (Weighted Loss):  {robust_acc*100:.2f}%")
print(f"Improvement:             {(robust_acc - baseline_acc)*100:.2f}%")
print("="*40)


--- Training Baseline Model (Ignoring Confidence) ---


Epoch 1: 100%|██████████| 132/132 [00:01<00:00, 88.16it/s]


Epoch 1 Loss: 0.5595


Epoch 2: 100%|██████████| 132/132 [00:01<00:00, 98.40it/s]


Epoch 2 Loss: 0.3571


...

Epoch 8 Loss: 0.0199


Epoch 9: 100%|██████████| 132/132 [00:01<00:00, 106.46it/s]


Epoch 9 Loss: 0.0129


Epoch 10: 100%|██████████| 132/132 [00:01<00:00, 106.80it/s]


Epoch 10 Loss: 0.0089
Robust Test Accuracy: 65.50%

FINAL SUMMARY:
Baseline (Standard Loss): 65.30%
Robust (Weighted Loss):  65.50%
Improvement:             0.20%
